# episode_systemwide 

This notebook transforms systemwide HMIS event data from `event_systemwide` into episodes of homelessness, with the goal of tracking inflow, outflow, and active status across time. The resulting dataset supports by-name lists, performance metrics, and other analytic products used by [[ORGANIZATION_NAME]].

Data is pulled from Azure Blob Storage using DefaultAzureCredential, and includes:

- [[EVENT_DATA_PARQUET_FILENAME]]: Systemwide event data across services, enrollments, and more
- [[ENROLLMENTS_PARQUET_FILENAME]]: Enrollment records with project metadata

Data are then joined and cleaned. 

The core logic in this code processes the dataset to create episodes of homelessness—time periods during which a person is considered continuously homeless. It produces a structured output where each row corresponds to a person-month within an episode.

Episode segmentation is based on:

- Status change (e.g., from "Homeless" to "Housed")
- Time gaps between events exceeding a threshold (default: **30 days**) Adjust as the community desires


# Import Dependencies

Load required packages for data processing and Azure connections.


In [ ]:
import looker_sdk
from looker_sdk import api_settings

from io import StringIO
import pandas as pd
from azure.storage.blob import BlobServiceClient
from io import BytesIO
import json
from datetime import datetime, timedelta
from datetime import datetime
import numpy as np 
import io
from datetime import date
from datetime import datetime
import pyarrow


import os
from azure.storage.blob import BlobServiceClient
from azure.identity import DefaultAzureCredential



In [ ]:
# #Comment out in production. Use for development

# Import Looker SDK connection from shared script
import sys
sys.path.insert(0, '..')
from [[LOCAL_SCRIPT_NAME]] import sdk

In [ ]:
# #Add chunk to connect to Looker API in production

# class [[ORG_PREFIX]]_Looker_API_Settings(api_settings.ApiSettings):
#     def __init__(self, *args, **kw_args):
#         self.my_var = kw_args.pop("my_var")
#         super().__init__(*args, **kw_args)

#     def read_config(self) -> api_settings.SettingsConfig:
#         config = super().read_config()
#         # See api_settings.SettingsConfig for required fields.
#         if self.my_var == "set":
#             config["base_url"] = TokenLibrary.getSecretWithLS('[[AZURE_KEY_VAULT_NAME]]','[[LOOKER_BASE_URL_SECRET]]')
#             config["client_id"] = TokenLibrary.getSecretWithLS('[[AZURE_KEY_VAULT_NAME]]','[[LOOKER_CLIENT_ID_SECRET]]')
#             config["client_secret"] = TokenLibrary.getSecretWithLS('[[AZURE_KEY_VAULT_NAME]]','[[LOOKER_CLIENT_SECRET_SECRET]]')          
#         return config

# sdk = looker_sdk.init40(config_settings=[[ORG_PREFIX]]_Looker_API_Settings(my_var="set"))

# vars(vars(vars(sdk)["transport"])["settings"])["timeout"] = 300

Run "az login" in the terminal before running this chunk; select production

In [ ]:
#In development outside of azure

# Define a helper function to get a table from '[[AZURE_CONTAINER_NAME]]'
def get_table(file_name):
    # Define the storage account and container name
    storage_account_name = "[[AZURE_STORAGE_ACCOUNT_NAME]]"
    container_name = "[[AZURE_CONTAINER_NAME]]"
    
    # Use DefaultAzureCredential to authenticate with Microsoft Entra credentials
    credential = DefaultAzureCredential()
    
    # Construct the BlobServiceClient URL with the storage account name
    blob_service_client = BlobServiceClient(
        account_url=f"https://{storage_account_name}.blob.core.windows.net",
        credential=credential
    )
    
    # Create a BlobClient for the specified blob
    blob_client = blob_service_client.get_blob_client(container=container_name, blob=file_name)
    
    # Download the blob content into a stream
    download_stream = blob_client.download_blob()
    parquet_data = io.BytesIO(download_stream.readall())
    
    # Load the Parquet data into a Pandas DataFrame
    df = pd.read_parquet(parquet_data)
    
    # Return the dataframe
    return df

In [ ]:
# # For production in Azure

# # define a helper function to get a table from '[[AZURE_CONTAINER_NAME]]'
# def get_table(file_name):
#     # Define parameter fields
#     container_name = "[[AZURE_CONTAINER_NAME]]"

#     # Create a BlobServiceClient object using the connection string
#     connection_string = TokenLibrary.getSecretWithLS('[[AZURE_KEY_VAULT_NAME]]', '[[ADLS_CONNECTION_SECRET]]')
#     blob_service_client = BlobServiceClient.from_connection_string(connection_string)

#     # Create a BlobClient for the specified blob
#     blob_client = blob_service_client.get_blob_client(container=container_name, blob=file_name)

#     # Download the blob content into a stream
#     download_stream = blob_client.download_blob()
#     parquet_data = io.BytesIO(download_stream.readall())

#     # Load the Parquet data into a Pandas DataFrame
#     df = pd.read_parquet(parquet_data)

#     # Return the dataframe
#     return df

# Extract Data Tables from [[ORGANIZATION_NAME]] Data Hub

In [ ]:
# read in all_events 
df0 = get_table("[[EVENT_DATA_PARQUET_FILENAME]]")
df0['EnrollmentID'] = df0['EnrollmentID'].astype(str)
df0['EventDate'] = pd.to_datetime(df0['EventDate'], errors='coerce').dt.date

In [ ]:
# read in enrollments 
enrollments0 = get_table("[[ENROLLMENTS_PARQUET_FILENAME]]")
enrollments0['EnrollmentID'] = enrollments0['EnrollmentID'].astype(str)

Pull in information about assessments to flag whether episodes had active assessments later in the script.

In [ ]:
# Define your query

body = { 
"model": "[[LOOKER_MODEL_NAME]]", 
"view": "client", 
"fields":[
  "clients.personal_id",
  "client_assessments.assessment_date"
], 
"filters": {
             "client_assessments.ref_assessment" : "[[ASSESSMENT_ID]]" # this is the assessment id for [[ASSESSMENT_NAME]]
              }, 
"limit": -1
}  

################ 
# Get Response 
################ 

RESULT_FORMAT = "csv" 

# Run the inline query
result = sdk.run_inline_query( 
result_format=RESULT_FORMAT,  
body=body 
) 

# Convert to dataframe.
htt_assess = pd.read_csv(StringIO(result))

columnHeaders={ 
    "Personal ID": "PersonalID",
    "Client Assessments Assessment Date": 'AssessmentDate'
}

htt_assess.rename(columns=columnHeaders,inplace=True)
htt_assess['AssessmentDate'] = pd.to_datetime(htt_assess['AssessmentDate'], errors='coerce')

Pulling information about birth dates to add "aged out" information for youth

In [ ]:
# Pull birth dates from client_model (Coordinated Entry model)
# client_model only includes folks with assessments tied to CE enrollment

body = { 
    "model": "[[LOOKER_MODEL_NAME]]", 
    "view": "client_model", 
    "fields": [
        "client_model.personal_id",
        "client_model.birth_date"
    ], 
    "limit": -1
}  

RESULT_FORMAT = "csv" 

result = sdk.run_inline_query( 
    result_format=RESULT_FORMAT,  
    body=body 
) 

birth_dates = pd.read_csv(StringIO(result))

## Prep Data

In [ ]:
# Join all_events with enrollments

# Drop overlapping columns from enrollments0 except the join key
overlap_cols = df0.columns.intersection(enrollments0.columns).difference(['EnrollmentID'])
enrollments_clean = enrollments0.drop(columns=overlap_cols)

# Merge cleanly
df = df0.merge(enrollments_clean, on='EnrollmentID', how='inner')


# The join is an inner join rather than a left join to cleanup the <1% of cases where the events brought in don't have a matching enrollment. 
# Mostly this is when service attendance dates (also some CLS and move-in dates) erroneously happpen outside of the enrollment period and are before the cutoff date of 7 years ago. 
# Sometimes there are erroneous events in non-participating agencies that are filtered out of the enrollments table.
# It's important to get the timing of the pipelines right so that the join is as complete as possible.


In [ ]:

# Filter out events that are older than 7 years
# revisit this when thinking about wanting to show inflow... might want to include older dates? 

cutoff_year = datetime.now().year - 7
cutoff_date = date(cutoff_year, 1, 1)
df = df[df['EventDate'] >= cutoff_date]

In [ ]:
# Select the columns needed for the final output

columns_needed = [
    'PersonalID',
    'EventDate',
    'ClientStatus',
    'ClientShelterStatus',
    'AgeTierAtEnrollment',
    'HouseholdType',
    'YYA',
    'IndividualChronicallyHomelessAtEnrollmentStart',
    'HeadOfHousehold',
    'CountAdults',
    'CountChildren'
]

df = df[columns_needed].copy()

# Make sure date is typed correctly
df['EventDate'] = pd.to_datetime(df['EventDate'])

In [ ]:
# Ensure consistent types across key columns
df['AgeTierAtEnrollment'] = df['AgeTierAtEnrollment'].astype(str)
df['IndividualChronicallyHomelessAtEnrollmentStart'] = df['IndividualChronicallyHomelessAtEnrollmentStart'].astype(str)
df['ClientStatus'] = df['ClientStatus'].astype(str)
df['HeadOfHousehold'] = df['HeadOfHousehold'].astype(str)
df['HouseholdType'] = df['HouseholdType'].astype(str)
df['YYA'] = df['YYA'].astype(str)
df['ClientShelterStatus'] = df['ClientShelterStatus'].astype(str)

# Generate Episodes

This code chunk identifies "episodes" of homelessness for individuals using event-level client status data.

Episodes are defined as continuous periods of the same status (e.g., "Homeless") with no more than a gap of `episode_gap_days` between relevant events.

The script also determines **inflow** and **outflow** types—classifying whether someone became newly homeless, returned from housing or inactivity, or exited due to housing, death, or inactivity.

Each episode is then **scaffolded across time** by generating a row for **each calendar month** in which the episode is active. This enables reporting based on specific **reporting months or periods**, regardless of the actual start and end dates of an episode.

The final output is a long-form table where **each row represents one episode-month for a person**, enriched with demographic and household flags. This scaffolding structure supports flexible reporting across monthly, quarterly, and yearly timeframes, and is designed to feed analytic tables and dashboards that monitor system performance over time.



In [ ]:
# ------------------------------
# Parameters
# ------------------------------
episode_gap_days = 30

# Cache today's date once (avoid repeated calls)
today = pd.Timestamp.today().normalize()
today_date = today.date()

# ------------------------------
# Functions to classify inflow and outflow types
# ------------------------------
def determine_inflow_type(current_status, last_status, last_reason, last_episode_end, current_date):
    if current_status != "Homeless":
        return None
    if last_status == "Housed":
        gap = (current_date - last_episode_end).days if last_episode_end else 0
        return "Newly Homeless" if gap >= 730 else "Return from Housed"
    if last_status == "Homeless" and last_reason == "inactive":
        gap = (current_date - last_episode_end).days if last_episode_end else 0
        return "Newly Homeless" if gap >= 730 else "Return from Inactive"
    return "Newly Homeless"

def determine_outflow_type(episode_status, reason, next_status):
    if episode_status != 'Homeless':
        return None
    if reason == 'inactive':
        return 'Inactive'
    if pd.isna(next_status):
        return 'Active'
    if next_status == 'Deceased':
        return 'Deceased'
    if next_status == 'Housed':
        return 'Permanently Housed'  
    return 'Active'

# ------------------------------
# Pre-sort data once (avoid sorting each group)
# ------------------------------
df = df.sort_values(['PersonalID', 'EventDate']).reset_index(drop=True)

# ------------------------------
# Main processing loop: per-person grouping
# ------------------------------
episode_time_rows = []

for person_id, group in df.groupby('PersonalID', sort=False):
    # Group is already sorted due to pre-sort
    events = list(group.itertuples(index=False))
    n_events = len(events)
    episode_index = 1

    # Person-level attributes (computed once per person)
    age_series = group['AgeTierAtEnrollment'].dropna()
    first_age_tier = age_series.iloc[0] if len(age_series) > 0 else None
    episode_chronic_status = "Yes" if (group['IndividualChronicallyHomelessAtEnrollmentStart'] == "Yes").any() else "No or Unknown"

    # Initialize first episode
    episode_start = events[0].EventDate
    episode_status = events[0].ClientStatus
    last_eventdate = episode_start
    
    # Track episode events as simple lists (avoid DataFrame creation until needed)
    ep_households = [events[0].HouseholdType]
    ep_yya = [events[0].YYA]
    ep_hohs = [events[0].HeadOfHousehold]
    ep_adults = [events[0].CountAdults]
    ep_children = [events[0].CountChildren]
    ep_shelters = [(events[0].EventDate, events[0].ClientShelterStatus)]

    last_status = None
    last_reason = None
    last_episode_end = None

    def process_episode(episode_start, episode_end, episode_status, ep_households, ep_yya, ep_hohs, ep_adults, ep_children, ep_shelters, episode_index, last_status, last_reason, last_episode_end, reason, next_status):
        """Process a completed episode and append monthly rows."""
        if episode_status != "Homeless":
            return
        
        # Compute episode-level aggregates from lists (faster than DataFrame)
        from collections import Counter
        hh_counts = Counter(ep_households)
        most_common_household = hh_counts.most_common(1)[0][0] if hh_counts else None
        
        has_yya = 'Yes' in ep_yya
        has_family = 'Household with Children and Adults' in ep_households
        has_adults = 'Household without Children' in ep_households
        has_hoh = 'Yes' in ep_hohs
        max_adults = max((x for x in ep_adults if pd.notna(x)), default=None)
        max_children = max((x for x in ep_children if pd.notna(x)), default=None)
        
        inflow_type = determine_inflow_type(episode_status, last_status, last_reason, last_episode_end, episode_start)
        outflow_type = determine_outflow_type(episode_status, reason, next_status)
        episode_id = f"{person_id}EP{episode_start.strftime('%y%m%d')}{episode_end.strftime('%y%m%d')}"
        
        # Sort shelter events once for this episode
        valid_shelters = [(d, s) for d, s in ep_shelters if pd.notna(s)]
        valid_shelters.sort(key=lambda x: x[0])
        
        # Generate monthly rows
        start_month = episode_start.replace(day=1)
        end_month = episode_end.replace(day=1)
        months = pd.date_range(start=start_month, end=end_month, freq='MS')
        
        for month_start in months:
            month_end = month_start + pd.offsets.MonthEnd(0)
            
            # Find last shelter status up to month_end
            last_shelter = None
            for d, s in valid_shelters:
                if d <= month_end:
                    last_shelter = s
                else:
                    break
            
            episode_time_rows.append({
                'EpisodeID': episode_id,
                'EpisodeIndex': episode_index,
                'PersonalID': person_id,
                'EpisodeStartDate': episode_start,
                'EpisodeEndDate': episode_end,
                'TimePeriodStartDate': month_start.date(),
                'TimePeriodEndDate': month_end.date(),
                'EpisodeOutflowType': outflow_type,
                'EpisodeInflowType': inflow_type,
                'EpisodeChronicStatus': episode_chronic_status,
                'DataAsOfDate': today_date,
                'LengthOfTimeExperiencingHomelessness': (month_end - episode_start).days + 1,
                'EpisodeAgeTierAtEntry': first_age_tier,
                'EpisodeHouseholdType': most_common_household,
                'LastShelterStatusInTimeframe': last_shelter,
                'FlagYYA': int(has_yya),
                'FlagFamilyWithChildren': int(has_family),
                'FlagAdults': int(has_adults),
                'FlagHeadOfHousehold': int(has_hoh),
                'EpisodeMaxCountAdults': max_adults,
                'EpisodeMaxCountChildren': max_children
            })

    # Process events
    for i in range(1, n_events):
        event = events[i]
        current_date = event.EventDate
        current_status = event.ClientStatus
        day_diff = (current_date - last_eventdate).days
        
        # Episode continues if status unchanged and gap is small
        if current_status == episode_status and day_diff < episode_gap_days:
            last_eventdate = current_date
            ep_households.append(event.HouseholdType)
            ep_yya.append(event.YYA)
            ep_hohs.append(event.HeadOfHousehold)
            ep_adults.append(event.CountAdults)
            ep_children.append(event.CountChildren)
            ep_shelters.append((event.EventDate, event.ClientShelterStatus))
            continue

        # Close current episode
        reason = "status change" if current_status != episode_status else "inactive"
        next_status = current_status if reason == "status change" else None

        if reason == "inactive":
            episode_end = last_eventdate + timedelta(days=episode_gap_days)
        elif next_status in ["Housed", "Deceased"]:
            episode_end = current_date
        else:
            episode_end = today

        process_episode(episode_start, episode_end, episode_status, ep_households, ep_yya, ep_hohs, 
                       ep_adults, ep_children, ep_shelters, episode_index, last_status, 
                       last_reason, last_episode_end, reason, next_status)

        # Prepare for next episode
        episode_index += 1
        last_status = episode_status
        last_reason = reason
        last_episode_end = episode_end

        episode_start = current_date
        episode_status = current_status
        last_eventdate = current_date
        ep_households = [event.HouseholdType]
        ep_yya = [event.YYA]
        ep_hohs = [event.HeadOfHousehold]
        ep_adults = [event.CountAdults]
        ep_children = [event.CountChildren]
        ep_shelters = [(event.EventDate, event.ClientShelterStatus)]

    # Handle final episode
    final_date = events[-1].EventDate
    reason = "inactive" if (today - final_date).days > episode_gap_days else "last row"
    
    if reason == "inactive":
        episode_end = last_eventdate + timedelta(days=episode_gap_days)
    else:
        episode_end = today

    process_episode(episode_start, episode_end, episode_status, ep_households, ep_yya, ep_hohs,
                   ep_adults, ep_children, ep_shelters, episode_index, last_status,
                   last_reason, last_episode_end, reason, None)

# Final output
final_df = pd.DataFrame(episode_time_rows)

In [ ]:
# Add AgedOutOfYYA column - date of 25th birthday if it falls within the episode
# Rename birth_dates columns for consistency
birth_dates.columns = ['PersonalID', 'BirthDate']
birth_dates['BirthDate'] = pd.to_datetime(birth_dates['BirthDate'], errors='coerce')

def calculate_25th_birthday(birth_date):
    """Calculate 25th birthday, handling leap day births by using March 1st."""
    if pd.isna(birth_date):
        return None
    # Check if born on Feb 29 (leap day)
    if birth_date.month == 2 and birth_date.day == 29:
        # Use March 1st for the 25th birthday
        return pd.Timestamp(year=birth_date.year + 25, month=3, day=1)
    else:
        return pd.Timestamp(year=birth_date.year + 25, month=birth_date.month, day=birth_date.day)

# Merge birth dates into final_df
final_df = final_df.merge(birth_dates, on='PersonalID', how='left')

# Calculate 25th birthday for each person
final_df['Birthday25'] = final_df['BirthDate'].apply(calculate_25th_birthday)

# Convert episode dates to datetime for comparison
final_df['EpisodeStartDate'] = pd.to_datetime(final_df['EpisodeStartDate'], errors='coerce')
final_df['EpisodeEndDate'] = pd.to_datetime(final_df['EpisodeEndDate'], errors='coerce')

# Set AgedOutOfYYA to the 25th birthday date if it falls within the episode, otherwise null
final_df['AgedOutOfYYA'] = final_df.apply(
    lambda row: row['Birthday25'] if (
        pd.notna(row['Birthday25']) and 
        pd.notna(row['EpisodeStartDate']) and 
        pd.notna(row['EpisodeEndDate']) and
        row['EpisodeStartDate'] <= row['Birthday25'] <= row['EpisodeEndDate']
    ) else None,
    axis=1
)

# Drop helper columns
final_df = final_df.drop(columns=['BirthDate', 'Birthday25'])

Ensure date types are specified

In [ ]:
date_cols = [
    "EpisodeStartDate",
    "EpisodeEndDate",
    "TimePeriodStartDate",
    "TimePeriodEndDate",
    "DataAsOfDate",
    "AgedOutOfYYA"
]

for col in date_cols:
    final_df[col] = pd.to_datetime(final_df[col], errors="coerce").dt.strftime("%Y-%m-%d")

# Flag episodes that overlap with a Coordinated Entry enrollment
An episode overlaps with a CE enrollment if:
   EpisodeStartDate <= ProjectExitDate (or ProjectExitDate is null) AND ProjectStartDate <= EpisodeEndDate

In [ ]:
# Get CE enrollments with relevant columns
ce_enrollments = enrollments0[enrollments0['ProgramID'] == [[CE_PROGRAM_ID]]][  # ProgramID for [[CE_PROGRAM_NAME]]
    ['PersonalID', 'ProjectStartDate', 'ProjectExitDate']
].copy()

# Ensure date types
ce_enrollments['ProjectStartDate'] = pd.to_datetime(ce_enrollments['ProjectStartDate'], errors='coerce')
ce_enrollments['ProjectExitDate'] = pd.to_datetime(ce_enrollments['ProjectExitDate'], errors='coerce')

# Merge episodes with CE enrollments on PersonalID
merged = (final_df[['PersonalID', 'EpisodeID', 'EpisodeStartDate', 'EpisodeEndDate']
                         ].copy()
                         .drop_duplicates()
                         .merge(ce_enrollments, on='PersonalID', how='left'))

# Check for overlap: episode overlaps CE if start <= exit (or exit is null) AND project_start <= episode_end
merged['overlaps'] = (
    ((merged['EpisodeStartDate'] <= merged['ProjectExitDate']) | merged['ProjectExitDate'].isna()) &
    (merged['ProjectStartDate'] <= merged['EpisodeEndDate'])
)

# Aggregate: CEActive is True if any CE enrollment overlaps with the episode
ce_active = merged.groupby('EpisodeID')['overlaps'].any().reset_index(name='CEActive')

# ------------------------------
# Flag whether an assessment was present during the CE enrollment period within the episode
# AssessmentPresent = True if AssessmentDate >= ProjectStartDate AND AssessmentDate <= EpisodeEndDate
# ------------------------------

# Merge assessments into the merged dataframe (which has episode + CE enrollment data)
merged_with_assess = merged.merge(htt_assess, on='PersonalID', how='left')

# Check if assessment falls within CE enrollment start and episode end
# Only consider rows where there was an overlapping CE enrollment
merged_with_assess['has_assessment'] = (
    merged_with_assess['overlaps'] &  # Only for overlapping CE enrollments
    merged_with_assess['AssessmentDate'].notna() &
    (merged_with_assess['AssessmentDate'] >= merged_with_assess['ProjectStartDate']) &
    (merged_with_assess['AssessmentDate'] <= merged_with_assess['EpisodeEndDate'])
)

# Aggregate: AssessmentPresent is True if any valid assessment exists for this episode
assessment_present = merged_with_assess.groupby('EpisodeID')['has_assessment'].any().reset_index(name='AssessmentPresent')

# Join back to final_df using episodeID

final_df = (final_df
        .merge(ce_active, on='EpisodeID', how='left')
        .merge(assessment_present, on='EpisodeID', how='left'))

# Join in Client Unique Identifiers

These are dropped initially because some PersonalIDs have multiple Client Unique Identifiers. These create issues in the episode create logic. So we join back in on PersonalID. Some Episodes will be duplicated for someone who has one PersonalID but multiple ClientUniqueIdentifiers. 

In [ ]:
final_df = final_df.merge(df0[['PersonalID', 'ClientUniqueIdentifier']].drop_duplicates(), on='PersonalID', how='left')

In [ ]:
# #For development outside of azure

# import io
# from azure.storage.blob import BlobServiceClient
# from azure.identity import DefaultAzureCredential

# # Define a function to write a DataFrame to Azure Blob Storage as a Parquet file
# def write_table(df, file_name):
#     # Define the storage account and container name
#     storage_account_name = "[[AZURE_STORAGE_ACCOUNT_NAME]]"
#     container_name = "[[AZURE_CONTAINER_NAME]]"

#     # Authenticate using Microsoft Entra credentials
#     credential = DefaultAzureCredential()

#     # Create a BlobServiceClient
#     blob_service_client = BlobServiceClient(
#         account_url=f"https://{storage_account_name}.blob.core.windows.net",
#         credential=credential
#     )

#     # Create a BlobClient for the specified blob
#     blob_client = blob_service_client.get_blob_client(container=container_name, blob=file_name)

#     # Convert DataFrame to Parquet format in memory
#     parquet_buffer = io.BytesIO()
#     df.to_parquet(parquet_buffer, engine="pyarrow", index=False)

#     # Upload the Parquet data to Azure Blob Storage
#     blob_client.upload_blob(parquet_buffer.getvalue(), overwrite=True)

# # Write the 'final' DataFrame to Blob Storage
# write_table(final_df, "[[OUTPUT_PARQUET_FILENAME]]")

In [ ]:
# #For Production inside  azure

# # Initiate Parameter Fields.
# container_name = "[[AZURE_CONTAINER_NAME]]"
# file_name = "[[OUTPUT_PARQUET_FILENAME]]"

# # Create a BlobServiceClient object using the connection string
# connection_string = TokenLibrary.getSecretWithLS('[[AZURE_KEY_VAULT_NAME]]', '[[ADLS_CONNECTION_SECRET]]')
# blob_service_client = BlobServiceClient.from_connection_string(connection_string)

# # Create a BlobClient for the final blob
# final_blob_client = blob_service_client.get_blob_client(container=container_name, blob=file_name)

# # Save DataFrame to a temporary Parquet file on Azure Blob Storage
# with BytesIO() as temp_buffer:
#     final_df.to_parquet(temp_buffer, engine='pyarrow', index=False)
#     final_blob_client.upload_blob(temp_buffer.getvalue(), overwrite=True)